In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, torch, math, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
from tqdm import tqdm
from typing import List, Tuple, Dict, Any, Optional, IO, BinaryIO, Tuple
from torch import Tensor, multinomial
from einops import reduce, einsum, rearrange
from jaxtyping import Float, Bool, Int

from vllm import LLM, SamplingParams
from openai import OpenAI

from cs336_alignment.math_baseline import evaluate_vllm

INFO 03-04 22:53:42 __init__.py:190] Automatically detected platform cuda.


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

### Examples in Using vllm

In [ ]:
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0,
    max_tokens=1024,
    stop=['\n']
)

# create an LLM instance. We can also use other models.
llm = LLM(model = 'Qwen/Qwen2.5-Math-1.5B')

# generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
# We can use output.outputs[0].text to get the generated text.
outputs = llm.generate(prompts, sampling_params)

# print the generated texts
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt}")
    print(f"Generated text: {generated_text}")
    print("-"*100)
    

INFO 03-03 15:38:56 config.py:542] This model supports multiple tasks: {'generate', 'score', 'embed', 'classify', 'reward'}. Defaulting to 'generate'.
INFO 03-03 15:38:56 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='Qwen/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='Qwen/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=Qwen/Qwen2.5-Math-1.5B, num_scheduler_st

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-03 15:39:00 model_runner.py:1115] Loading model weights took 2.8797 GB
INFO 03-03 15:39:00 worker.py:267] Memory profiling takes 0.54 seconds
INFO 03-03 15:39:00 worker.py:267] the current vLLM instance can use total_gpu_memory (79.25GiB) x gpu_memory_utilization (0.90) = 71.32GiB
INFO 03-03 15:39:00 worker.py:267] model weights take 2.88GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 66.96GiB.
INFO 03-03 15:39:01 executor_base.py:110] # CUDA blocks: 156715, # CPU blocks: 9362
INFO 03-03 15:39:01 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 612.17x
INFO 03-03 15:39:03 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:14<00:00,  2.34it/s]

INFO 03-03 15:39:18 model_runner.py:1562] Graph capturing finished in 15 secs, took 0.19 GiB
INFO 03-03 15:39:18 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 18.75 seconds



Processed prompts: 100%|██████████| 4/4 [00:00<00:00, 10.90it/s, est. speed input: 60.17 toks/s, output: 402.06 toks/s]

Prompt: Hello, my name is
Generated text:  K deadly. I have to find out what I fall into as one of Dumbass Meets Us Programming Challenges on CodeChef.
----------------------------------------------------------------------------------------------------
Prompt: The president of the United States is
Generated text:  planning to distribute a certain number of kits to schools around the world to enhance their understanding of international relations. Each kit contains materials for a discussion on how to handle cyber-attacks. If the president decides to distribute 300 kits and wants to ensure each school gets an equal number of kits, how many kits will each school receive if there are 6 schools participating?
----------------------------------------------------------------------------------------------------
Prompt: The capital of France is
Generated text:  Paris. In which month are many people celebrating their birthdays on Republic Day in India?
------------------------------------------

### Zero-shot performance on the MATH dataset.

#### Dataset

- Since the original version of the MATH-12.5k dataset is not available online, we use
    - math12k dataset in [math12k](https://huggingface.co/datasets/hiyouga/math12k) with 500 evaluation data and 12k training data.
    - GSM8k dataset in [gsm8k](https://huggingface.co/datasets/openai/gsm8k) with 7.47k training data and 1.32k evaluation data.
- We use `vllm.generate` function with `temperature=1.0, top_p=1.0, max_tokens=1024, stop=['</answer>'], include_stop_str_in_output=True`.
- We use R1zero's prompt (notice that this is not the optimal prompt based on Dr.GRPO paper. Directly state the problem and ask the model to continue will give you an even better result. But this prompt is suitable for RL since the reward will rise quickly and it will be easier to verify the answer).
- The format reward is $0.1480$ and the answer reward is $0.0240$.
- Greedy decoding gives a format reward of $0.3800$ and an answer reward of $0.1460$.
- Many wrong answer are because of the format mismatch, which is because we use R1-Zero prompt. So the accuracy is much lower than what Qwen reported.

In [ ]:
# data preprocessing
train_df = pd.read_parquet("/home/ruiqizhang/CS336/assignment5-alignment/data/math/train-00000-of-00001.parquet")
test_df = pd.read_parquet("/home/ruiqizhang/CS336/assignment5-alignment/data/math/test-00000-of-00001.parquet")

train_data = []
test_data = []
for i, row in train_df.iterrows():
    train_data.append({
        'question': row['problem'],
        'answer': str(row['answer']),
    })
for i, row in test_df.iterrows():
    test_data.append({
        'question': row['problem'],
        'answer': row['answer'],
    })

train_data_path = "/home/ruiqizhang/CS336/assignment5-alignment/data/math/train.json"
test_data_path = "/home/ruiqizhang/CS336/assignment5-alignment/data/math/test.json"

with open(train_data_path, 'w') as f:
    for obj in train_data:
        json.dump(obj, f)
        f.write('\n')

with open(test_data_path, 'w') as f:
    for obj in test_data:
        json.dump(obj, f)
        f.write('\n')

### SFT

- Since the SFT data in the assignment is not available to us, we use DeepSeek-R1 to generate the SFT data. To generate such data, check `deepseek_r1_completion.py`.
- We generate up to 3 responses for every question in MATH training set (12k) from deepSeek-R1 and we get 300 questions with no correct answers and 11.7k questions with correct answers. This costs 120 CNY (about 17 dollars).
- We use the 12000 samples (including incorrect ones) to do SFT. We use `lr = 1e-5, wd = 0.0`. We use a global batch size of 16 and a gradient accumulation step of 4. We train 1 epoch. It takes about 40 min - 1 hour. **At the 100-th step (global), the average reward at validation reaches 0.5** (with `temp=1.0` at validation time) (strict format reward. Only allow one single space between `</think>` and `<answer>`).

### Expert Iteration

- Implement the Expert Iteration framework. See `expert_iteration.py`.

### GRPO

- Implement the GRPO framework. See `rl_utils.py` and `rl_grpo.py`.


In [15]:
r1_prompts_path = "/home/ruiqizhang/CS336/assignment5-alignment/cs336_alignment/prompts/r1_zero.prompt"
with open(r1_prompts_path, 'r') as f:
    r1_prompts = f.read()

os.environ['DEEPSEEK_API_KEY'] = 'sk-d157e7c624b644498861a1210bfd5a47'

client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com")

question_raw = "If $5x - 3 = 12$, what is the value of $5x + 3$?"
question = r1_prompts.format(question=question_raw)
print(question)

A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: If $5x - 3 = 12$, what is the value of $5x + 3$?
Assistant: <think>


In [19]:
type(response)

openai.types.chat.chat_completion.ChatCompletion

In [17]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": r1_prompts},
        {"role": "user", "content": f"{question_raw}"},
    ],
    stream=False
)

print(response.choices[0].message.content)

We are given the equation: \(5x - 3 = 12\).  
We need to find \(5x + 3\).

First, solve for \(5x\) from the given equation:  
\(5x - 3 = 12\)  
Add 3 to both sides:  
\(5x = 15\).

Now, we want \(5x + 3\).  
Since \(5x = 15\),  
\(5x + 3 = 15 + 3 = 18\).

Thus, the value is \(18\).

</think>
<answer>18</answer>


In [ ]:
response.usage.completion_tokens, response.usage.prompt_tokens, response.usage.prompt_tokens_details.cached_tokens # This is to compute the cost.

(11, 10, 0)

In [9]:
response

ChatCompletion(id='16b397f8-eb3e-4219-83bb-6961f8a9f9d2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1772588498, model='deepseek-chat', object='chat.completion', service_tier=None, system_fingerprint='fp_eaab8d114b_prod0820_fp8_kvcache', usage=CompletionUsage(completion_tokens=11, prompt_tokens=10, total_tokens=21, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=10))

In [ ]:
# Filter the correct data.

sft_train_data_path = "/home/ruiqizhang/CS336/assignment5-alignment/data/math/sft_train.json"
sft_train_data_correct_path = "/home/ruiqizhang/CS336/assignment5-alignment/data/math/sft_train_correct.json"

correct_data = []
with open(sft_train_data_path, "r") as f:
    data = [json.loads(line) for line in f]
#     for obj in data:
#         if obj['correct']:
#             correct_data.append(obj)

# with open(sft_train_data_correct_path, 'w') as f:
#     for obj in correct_data:
#         json.dump(obj, f)
#         f.write('\n')